# Grad-CAM Explainer with ResNet50

This notebook classifies an image with a pretrained ResNet50 and overlays a Grad-CAM heatmap for the highest-scoring class.

**Resources:** [ResNet50 weights](https://pytorch.org/vision/stable/models/generated/torchvision.models.resnet50.html) · [ImageNet-1K dataset](https://huggingface.co/datasets/ILSVRC/imagenet-1k) · [Grad-CAM paper](https://arxiv.org/abs/1610.02391) · [Source code](https://github.com/ChadSaglam/gradcam-explainer)

The model uses torchvision's `IMAGENET1K_V2` weights. No training or ImageNet download is required. Internet access is needed for the package, model weights and sample image; a GPU is optional.

## 1. Setup

Install the Grad-CAM package. PyTorch, torchvision, Pillow and Matplotlib are already available in the Kaggle image.

In [ ]:
!pip install -q grad-cam

In [ ]:
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from torchvision import models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 2. Load the model

`ResNet50_Weights.DEFAULT` currently refers to `IMAGENET1K_V2`. The weights also provide the matching input transform and the 1,000 class labels. Grad-CAM reads activations and gradients from the final residual block.

In [ ]:
WEIGHTS = models.ResNet50_Weights.DEFAULT
PREPROCESS = WEIGHTS.transforms()
LABELS = WEIGHTS.meta["categories"]

model = models.resnet50(weights=WEIGHTS).eval().to(DEVICE)
cam = GradCAM(model=model, target_layers=[model.layer4[-1]])

print(f"Weights: {WEIGHTS.name}")
print(f"Classes: {len(LABELS)}")

## 3. Define the explanation step

The same preprocessed crop is used for classification and for the background of the heatmap. This keeps the overlay aligned when the input is not square.

In [ ]:
def explain(image):
    image = image.convert("RGB")
    input_tensor = PREPROCESS(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probabilities = model(input_tensor).softmax(dim=1)[0]

    top = probabilities.topk(5)
    heatmap = cam(input_tensor=input_tensor, targets=None)[0]

    mean = input_tensor.new_tensor(PREPROCESS.mean).view(3, 1, 1)
    std = input_tensor.new_tensor(PREPROCESS.std).view(3, 1, 1)
    base = (input_tensor[0] * std + mean).permute(1, 2, 0)
    base = base.detach().cpu().numpy().clip(0, 1)
    overlay = show_cam_on_image(base, heatmap, use_rgb=True)

    return top, base, overlay

## 4. Run the sample

The sample photograph comes from the [PyTorch Hub examples](https://github.com/pytorch/hub/blob/master/images/dog.jpg). To use your own image, add it through Kaggle's **Add Input** panel and replace the download step with `Image.open("/kaggle/input/...")`.

In [ ]:
SAMPLE_URL = "https://github.com/pytorch/hub/raw/master/images/dog.jpg"
urllib.request.urlretrieve(SAMPLE_URL, "sample.jpg")
image = Image.open("sample.jpg")
top, base, overlay = explain(image)

for score, index in zip(top.values, top.indices):
    print(f"{LABELS[int(index)]}: {float(score):.3f}")

## 5. Show the heatmap

The target is the highest-scoring class. Warmer colors indicate regions with a larger contribution to that class score.

In [ ]:
target_name = LABELS[int(top.indices[0])]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(base)
axes[0].set_title("Preprocessed input")
axes[0].axis("off")
axes[1].imshow(overlay)
axes[1].set_title(f"Grad-CAM: {target_name}")
axes[1].axis("off")

plt.tight_layout()
plt.savefig("gradcam.png", dpi=150, bbox_inches="tight")
plt.show()

## Reading the result

Grad-CAM highlights spatial evidence for one class; it does not establish a causal explanation or rule out confounding features. Compare the overlay with the input and check whether the highlighted region contains the predicted object.

ResNet50 is limited to the 1,000 ImageNet categories. An image outside those categories can still receive a confident but misleading prediction. For a different class, pass a `ClassifierOutputTarget` from `pytorch_grad_cam.utils.model_targets` through the `targets` argument in `explain`.